# RNN + Attention + LSTM — One‑Stop Math + Deep Theory + Worked Examples (PyTorch)

Goal: Understand the “bridge” architecture path:
**Vanilla RNN → (problems) → Attention & Gating (solutions) → Transformers.**

This notebook includes:
- why vanilla RNN struggles
- attention math **with detailed explanation**
- an attention numeric example in PyTorch
- LSTM (gated neuron) math **with detailed explanation**
- a one-step LSTM numeric example
- comparison with `torch.nn.LSTMCell` so you learn PyTorch API too

## 1) Vanilla RNN recap + the specific limitation

Vanilla recurrence:
$$
a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h,\quad
h_t = \tanh(a_t)
$$

### Why long dependencies are hard
The gradient to an earlier hidden state involves a product:
$$
\frac{\partial L}{\partial h_{t-k}}
=
\frac{\partial L}{\partial h_t}
\prod_{i=t-k+1}^{t}
\frac{\partial h_i}{\partial h_{i-1}}.
$$

Each factor roughly contains \(W_{hh}\) and a nonlinearity derivative, so it can shrink/grow exponentially with \(k\).

**Practical symptom**: the model learns short patterns well, but struggles to “remember” information from far back in the sequence.

## 2) Attention on top of RNN (what it is, why it helps)

Assume you already computed a set of hidden states (often called “encoder states”):
$$
h_1, h_2, \dots, h_T,\quad h_s \in \mathbb{R}^{d_h}.
$$

Now at some time \(t\), you have a **query** representation \(q_t\).  
In simple RNN-attention, a common choice is \(q_t = h_t\).

### 2.1 Scores (similarity)
A simple dot-product score:
$$
e_{t,s} = q_t^\top h_s
$$

**Interpretation**
- \(e_{t,s}\) is a scalar measuring how relevant state \(h_s\) is to query \(q_t\).
- Larger score → more attention weight.

### 2.2 Attention weights (softmax)
$$
\alpha_{t,s} = \frac{\exp(e_{t,s})}{\sum_{k=1}^{T}\exp(e_{t,k})}
$$

**Interpretation**
- For a fixed \(t\), the weights \(\alpha_{t,1},\dots,\alpha_{t,T}\) sum to 1.
- It’s a probability distribution over positions \(s\) (where to “look”).

### 2.3 Context vector (weighted sum)
$$
c_t = \sum_{s=1}^{T} \alpha_{t,s} h_s
$$

**Interpretation**
- \(c_t\) is a summary of the whole sequence **focused** on what the query needs.

### Why attention helps gradients
Instead of forcing information from \(h_s\) to travel through a long recurrent chain to influence time \(t\), attention creates a short path: \(h_s \to c_t \to \text{loss}\).

So the model can access distant information **more directly**.

### Worked numerical example: compute scores → weights → context (PyTorch)

We create three hidden states in \(\mathbb{R}^2\), set \(q=h_3\), and compute:
- dot-product scores
- softmax weights
- context vector \(c\)

In [ ]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

# 3 hidden states in R^2
h1 = torch.tensor([[1.0],[0.0]])
h2 = torch.tensor([[0.0],[1.0]])
h3 = torch.tensor([[1.0],[1.0]])

H = [h1, h2, h3]
q = h3  # query

scores = torch.stack([ (q.T @ hs).squeeze() for hs in H ])  # (T,)
alpha  = F.softmax(scores, dim=0)                           # (T,)

c = sum(alpha[s] * H[s] for s in range(len(H)))             # (d_h,1)

print("scores e:", scores)
print("alpha weights:", alpha)
print("context c:\n", c)

## 3) Gated neurons: LSTM (what problem it solves)

Vanilla RNN “overwrites” its hidden state each step:
$$
h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)
$$
This makes it hard to preserve information over long ranges.

LSTM introduces a **separate memory channel** \(C_t\) (cell state) with an **additive update**.
This is the key architectural trick to fight vanishing gradients.

### 3.1 Gate idea
A gate is a sigmoid output in \([0,1]\):
$$
g = \sigma(\cdot)
$$
- \(g\approx 0\): block information
- \(g\approx 1\): pass information
- in-between: partial pass

So gates act like **learned valves**.

## 4) LSTM equations (with detailed interpretation)

Forget gate (how much old memory to keep):
$$
f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)
$$

Input gate (how much new information to write):
$$
i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)
$$

Candidate memory (new content proposal):
$$
\tilde C_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)
$$

Cell update (additive memory path):
$$
C_t = f_t \odot C_{t-1} + i_t \odot \tilde C_t
$$

Output gate (how much memory to expose as hidden state):
$$
o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)
$$

Hidden state:
$$
h_t = o_t \odot \tanh(C_t)
$$

### Why this helps gradients (the one-line reason)
Because:
$$
\frac{\partial C_t}{\partial C_{t-1}} = f_t
$$
If the model sets \(f_t \approx 1\), gradients can flow across many steps without shrinking rapidly.

### Worked numerical example: one LSTM step (PyTorch)

We pick tiny matrices and vectors so you can verify each gate manually.

In [ ]:
import torch

def sigmoid(x): 
    return torch.sigmoid(x)

torch.set_printoptions(precision=4, sci_mode=False)

d_x, d_h = 2, 2

x_t   = torch.tensor([[1.0],[2.0]])
h_prev= torch.tensor([[0.5],[-0.5]])
C_prev= torch.tensor([[1.0],[0.0]])

W_f = torch.tensor([[0.5, -0.2],[0.1, 0.4]])
U_f = torch.tensor([[0.3, 0.1],[-0.2, 0.2]])
b_f = torch.zeros((d_h,1))

W_i = torch.tensor([[0.4, 0.2],[-0.3, 0.1]])
U_i = torch.tensor([[0.2, -0.1],[0.1, 0.3]])
b_i = torch.zeros((d_h,1))

W_c = torch.tensor([[0.6, -0.1],[0.2, 0.5]])
U_c = torch.tensor([[0.1, 0.2],[0.3, -0.2]])
b_c = torch.zeros((d_h,1))

W_o = torch.tensor([[0.2, 0.3],[0.4, -0.2]])
U_o = torch.tensor([[0.1, -0.2],[0.2, 0.1]])
b_o = torch.zeros((d_h,1))

f_t = sigmoid(W_f @ x_t + U_f @ h_prev + b_f)
i_t = sigmoid(W_i @ x_t + U_i @ h_prev + b_i)
C_tilde = torch.tanh(W_c @ x_t + U_c @ h_prev + b_c)

C_t = f_t * C_prev + i_t * C_tilde
o_t = sigmoid(W_o @ x_t + U_o @ h_prev + b_o)
h_t = o_t * torch.tanh(C_t)

print("Forget gate f_t:\n", f_t)
print("Input gate  i_t:\n", i_t)
print("Candidate  C~_t:\n", C_tilde)
print("New cell   C_t:\n", C_t)
print("Output gate o_t:\n", o_t)
print("New hidden h_t:\n", h_t)

## 5) Same step using `torch.nn.LSTMCell` (API mapping to equations)

PyTorch packs all gates into one big matrix internally.

Gate order in `nn.LSTMCell` is **(i, f, g, o)** where \(g\) corresponds to \(\tilde C_t\).

We copy our weights into the packed representation so you can see that the module reproduces the manual math.

In [ ]:
from torch import nn

cell = nn.LSTMCell(input_size=d_x, hidden_size=d_h)

with torch.no_grad():
    weight_ih = torch.cat([W_i, W_f, W_c, W_o], dim=0)  # (4*d_h, d_x)
    weight_hh = torch.cat([U_i, U_f, U_c, U_o], dim=0)  # (4*d_h, d_h)
    bias = torch.cat([b_i, b_f, b_c, b_o], dim=0).squeeze(1)  # (4*d_h,)

    cell.weight_ih.copy_(weight_ih)
    cell.weight_hh.copy_(weight_hh)
    cell.bias_ih.copy_(bias)
    cell.bias_hh.zero_()

h_next, C_next = cell(x_t.T, (h_prev.T, C_prev.T))

print("nn.LSTMCell outputs")
print("h_next:\n", h_next.T)
print("C_next:\n", C_next.T)

print("\nDifferences vs manual:")
print("||h_next - h_t|| =", torch.norm(h_next.T - h_t).item())
print("||C_next - C_t|| =", torch.norm(C_next.T - C_t).item())

## 6) Summary: what attention and gating each fix

- **Attention**: shortens information/gradient paths by letting the model “look” anywhere in the sequence.
- **LSTM/GRU gating**: stabilizes memory over time using additive updates and learned gates.

These ideas set the stage for Transformers, which remove recurrence entirely and use attention as the main mechanism.